# Cohort Statistical Analysis

## Setup and Data Loading

In [1]:
import os
from contextlib import redirect_stdout
from itertools import combinations
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.anova import AnovaRM
from statsmodels.stats.multitest import multipletests

cur_dir = os.getcwd()
PROJECT_ROOT = cur_dir if os.path.isdir(os.path.join(cur_dir, "results")) else os.path.dirname(cur_dir)
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
OUTPUT_TXT_PATH = os.path.join(RESULTS_DIR, "statistical_analysis_results.txt")

df_comp = pd.read_csv(os.path.join(RESULTS_DIR, "cohort_survey_eeg_comparison.csv"))
df_stat = pd.read_csv(os.path.join(RESULTS_DIR, "cohort_stationary_summary.csv"))

# Merge Post-Swallow Clean Recovery Posterior Alpha/Beta Delta into df_comp
post_clean_ab_post = (
    df_stat[df_stat["Phase"] == "4b_Post_Swallow_Clean_Recovery"]
    [["Participant", "Condition", "Delta_Alpha_Beta_Post"]]
    .rename(columns={"Delta_Alpha_Beta_Post": "PostClean_Delta_AB_Post"})
)
df_comp = df_comp.merge(post_clean_ab_post, on=["Participant", "Condition"], how="left")

EEG_CONDS = ["eau", "the_vert", "biscuit", "chocolat"]
SURVEY_CONDS = ["the_vert", "biscuit", "chocolat"]

print(f"Loaded cohort_survey_eeg_comparison: {df_comp.shape}")
print(f"Loaded cohort_stationary_summary: {df_stat.shape}")
print(f"Merged PostClean_Delta_AB_Post non-null count: {df_comp['PostClean_Delta_AB_Post'].notna().sum()}")

Loaded cohort_survey_eeg_comparison: (32, 27)
Loaded cohort_stationary_summary: (336, 20)
Merged PostClean_Delta_AB_Post non-null count: 32


## Helper Functions

In [2]:
def check_normality(df, metric, conditions):
    print(f"\n--- Shapiro-Wilk Normality: {metric} ---")
    all_normal = True
    for cond in conditions:
        vals = df.loc[df["Condition"] == cond, metric].dropna()
        w_stat, p_val = stats.shapiro(vals)
        is_norm = p_val > 0.05
        all_normal = all_normal and is_norm
        status = "Normal" if is_norm else "NON-NORMAL"
        print(f"  {cond:10s}: W = {w_stat:.4f}, p = {p_val:.4f} ({status})")
    return all_normal


def run_omnibus_tests(df, metric, conditions, is_normal):
    sub = df[df["Condition"].isin(conditions)].dropna(subset=[metric]).copy()
    piv = sub.pivot(index="Participant", columns="Condition", values=metric)[conditions]
    if is_normal:
        aov = AnovaRM(data=sub, depvar=metric, subject="Participant", within=["Condition"]).fit()
        anova_row = aov.anova_table.iloc[0]
        f_val = anova_row["F Value"]
        df1 = int(anova_row["Num DF"])
        df2 = int(anova_row["Den DF"])
        p_rm = anova_row["Pr > F"]
        print(f"  [Parametric RM-ANOVA (Normal)] F({df1}, {df2}) = {f_val:.4f}, p = {p_rm:.4f}")
    else:
        chi2, p_fr = stats.friedmanchisquare(*[piv[c] for c in conditions])
        n_subj, k_cond = piv.shape
        kendall_w = chi2 / (n_subj * (k_cond - 1))
        print(f"  [Non-Parametric Friedman (Non-Normal)] chi2({k_cond - 1}) = {chi2:.4f}, p = {p_fr:.4f}, Kendall's W = {kendall_w:.4f}")
    return piv


def run_wilcoxon_vs_zero(df, metric, conditions):
    print(f"\n--- One-Sample Wilcoxon vs 0: {metric} ---")
    raw_p = []
    rows = []
    for cond in conditions:
        vals = df.loc[df["Condition"] == cond, metric].dropna()
        mean_val = vals.mean()
        sem_val = vals.std(ddof=1) / np.sqrt(len(vals)) if len(vals) > 1 else 0.0
        med_val = vals.median()
        iqr_val = vals.quantile(0.75) - vals.quantile(0.25)
        if np.all(vals == 0):
            w_stat, p_val = 0.0, 1.0
        else:
            w_stat, p_val = stats.wilcoxon(vals, alternative="two-sided")
        raw_p.append(p_val)
        rows.append((cond, mean_val, sem_val, med_val, iqr_val, w_stat, p_val))

    _, p_fdr, _, _ = multipletests(raw_p, method="fdr_bh")
    _, p_bonf, _, _ = multipletests(raw_p, method="bonferroni")
    for idx, (cond, mean_val, sem_val, med_val, iqr_val, w_stat, p_val) in enumerate(rows):
        print(
            f"  {cond:10s}: mean = {mean_val:+.4f}, SEM = {sem_val:.4f}, median = {med_val:+.4f}, IQR = {iqr_val:.4f}, "
            f"W = {w_stat:.2f}, p_raw = {p_val:.4f}, p_BH_FDR = {p_fdr[idx]:.4f}, p_Bonf = {p_bonf[idx]:.4f}"
        )


def run_pairwise_wilcoxon(piv, metric):
    pairs = list(combinations(piv.columns, 2))
    raw_p = []
    stats_list = []
    for c1, c2 in pairs:
        diff = piv[c1] - piv[c2]
        if np.all(diff == 0):
            w_stat, p_val = 0.0, 1.0
        else:
            w_stat, p_val = stats.wilcoxon(piv[c1], piv[c2], alternative="two-sided")
        raw_p.append(p_val)
        stats_list.append((c1, c2, w_stat, p_val, diff.mean(), diff.median()))

    _, p_fdr, _, _ = multipletests(raw_p, method="fdr_bh")
    _, p_bonf, _, _ = multipletests(raw_p, method="bonferroni")

    print(f"\n--- Post-Hoc Pairwise Wilcoxon: {metric} ---")
    for idx, (c1, c2, w_stat, p_val, m_diff, med_diff) in enumerate(stats_list):
        print(
            f"  {c1:10s} vs {c2:10s}: mean_diff = {m_diff:+.4f}, med_diff = {med_diff:+.4f}, "
            f"W = {w_stat:.2f}, p_raw = {p_val:.4f}, p_BH_FDR = {p_fdr[idx]:.4f}, p_Bonf = {p_bonf[idx]:.4f}"
        )


def analyse_metric(df, metric, conditions, test_zero=False):
    print("=" * 72)
    print(f"ANALYSIS: {metric}")
    print("=" * 72)
    is_normal = check_normality(df, metric, conditions)
    piv = run_omnibus_tests(df, metric, conditions, is_normal)
    if test_zero:
        run_wilcoxon_vs_zero(df, metric, conditions)
    run_pairwise_wilcoxon(piv, metric)


def run_band_surge_test(df_stat_in, band_col, band_label, conditions):
    print("=" * 72)
    print(f"PRE-TOUCH vs POST-SWALLOW: {band_label}")
    print("=" * 72)
    pre = df_stat_in[df_stat_in["Phase"] == "2_Pre_Touch_Rest"].set_index(["Participant", "Condition"])
    post = df_stat_in[df_stat_in["Phase"] == "4b_Post_Swallow_Clean_Recovery"].set_index(["Participant", "Condition"])

    raw_p = []
    rows = []
    for cond in conditions:
        v_pre = pre.xs(cond, level="Condition")[band_col].dropna()
        v_post = post.xs(cond, level="Condition")[band_col].dropna()
        common = v_pre.index.intersection(v_post.index)
        v_pre = v_pre.loc[common]
        v_post = v_post.loc[common]
        pct = ((v_post - v_pre) / v_pre * 100.0).median()
        diff = v_post - v_pre
        if np.all(diff == 0):
            w_stat, p_val = 0.0, 1.0
        else:
            w_stat, p_val = stats.wilcoxon(v_post, v_pre, alternative="two-sided")
        raw_p.append(p_val)
        rows.append((cond, len(common), v_pre.median(), v_post.median(), pct, w_stat, p_val))

    _, p_fdr, _, _ = multipletests(raw_p, method="fdr_bh")
    _, p_bonf, _, _ = multipletests(raw_p, method="bonferroni")
    for idx, (cond, n_obs, m_pre, m_post, pct, w_stat, p_val) in enumerate(rows):
        print(
            f"  {cond:10s} (N={n_obs}): med_pre={m_pre:.4f}, med_post={m_post:.4f}, "
            f"median_change={pct:+.4f}%, W={w_stat:.2f}, p_raw={p_val:.4f}, "
            f"p_BH_FDR={p_fdr[idx]:.4f}, p_Bonf={p_bonf[idx]:.4f}"
        )


def run_spearman_correlations(df_comp_in, survey_cols, eeg_cols):
    print("=" * 72)
    print("SPEARMAN RANK CORRELATIONS: SURVEY METRICS vs EEG DELTAS (Non-Water Conditions)")
    print("=" * 72)
    non_water = df_comp_in[df_comp_in["Condition"] != "eau"]
    raw_p = []
    rows = []
    for s_col in survey_cols:
        for e_col in eeg_cols:
            sub = non_water[[s_col, e_col]].dropna()
            rho, p_val = stats.spearmanr(sub[s_col], sub[e_col])
            raw_p.append(p_val)
            rows.append((s_col, e_col, len(sub), rho, p_val))

    _, p_fdr, _, _ = multipletests(raw_p, method="fdr_bh")
    _, p_bonf, _, _ = multipletests(raw_p, method="bonferroni")
    for idx, (s_col, e_col, n_obs, rho, p_val) in enumerate(rows):
        print(
            f"  {s_col:17s} vs {e_col:24s} (N={n_obs}): "
            f"rho = {rho:+.4f}, p_raw = {p_val:.4f}, "
            f"p_BH_FDR = {p_fdr[idx]:.4f}, p_Bonf = {p_bonf[idx]:.4f}"
        )

## Frontal Alpha Asymmetry Delta (PostClean_Delta_FAA_F4_F3)

In [3]:
analyse_metric(df_comp, "PostClean_Delta_FAA_F4_F3", EEG_CONDS, test_zero=True)

ANALYSIS: PostClean_Delta_FAA_F4_F3

--- Shapiro-Wilk Normality: PostClean_Delta_FAA_F4_F3 ---
  eau       : W = 0.9454, p = 0.6652 (Normal)
  the_vert  : W = 0.9308, p = 0.5231 (Normal)
  biscuit   : W = 0.9284, p = 0.5015 (Normal)
  chocolat  : W = 0.7826, p = 0.0186 (NON-NORMAL)
  [Non-Parametric Friedman (Non-Normal)] chi2(3) = 4.9500, p = 0.1755, Kendall's W = 0.2062

--- One-Sample Wilcoxon vs 0: PostClean_Delta_FAA_F4_F3 ---
  eau       : mean = -0.0069, SEM = 0.0429, median = +0.0245, IQR = 0.1391, W = 17.00, p_raw = 0.9453, p_BH_FDR = 0.9453, p_Bonf = 1.0000
  the_vert  : mean = +0.1079, SEM = 0.0685, median = +0.0895, IQR = 0.1809, W = 7.00, p_raw = 0.1484, p_BH_FDR = 0.5000, p_Bonf = 0.5938
  biscuit   : mean = +0.0412, SEM = 0.1396, median = +0.0413, IQR = 0.2859, W = 13.00, p_raw = 0.5469, p_BH_FDR = 0.7292, p_Bonf = 1.0000
  chocolat  : mean = -0.0383, SEM = 0.0302, median = -0.0578, IQR = 0.0444, W = 9.00, p_raw = 0.2500, p_BH_FDR = 0.5000, p_Bonf = 1.0000

--- Post-Hoc 

## Frontocentral Alpha-Beta Ratio Delta (PostClean_Delta_AB_FC)

In [4]:
analyse_metric(df_comp, "PostClean_Delta_AB_FC", EEG_CONDS, test_zero=True)

ANALYSIS: PostClean_Delta_AB_FC

--- Shapiro-Wilk Normality: PostClean_Delta_AB_FC ---
  eau       : W = 0.6880, p = 0.0016 (NON-NORMAL)
  the_vert  : W = 0.9318, p = 0.5323 (Normal)
  biscuit   : W = 0.6305, p = 0.0004 (NON-NORMAL)
  chocolat  : W = 0.9230, p = 0.4548 (Normal)
  [Non-Parametric Friedman (Non-Normal)] chi2(3) = 5.5500, p = 0.1357, Kendall's W = 0.2312

--- One-Sample Wilcoxon vs 0: PostClean_Delta_AB_FC ---
  eau       : mean = +0.1427, SEM = 0.0799, median = +0.0978, IQR = 0.0818, W = 3.00, p_raw = 0.0391, p_BH_FDR = 0.0781, p_Bonf = 0.1562
  the_vert  : mean = +0.0662, SEM = 0.1043, median = +0.1323, IQR = 0.2320, W = 11.00, p_raw = 0.3828, p_BH_FDR = 0.3828, p_Bonf = 1.0000
  biscuit   : mean = +0.3705, SEM = 0.1962, median = +0.2536, IQR = 0.1680, W = 2.00, p_raw = 0.0234, p_BH_FDR = 0.0781, p_Bonf = 0.0938
  chocolat  : mean = +0.2670, SEM = 0.1910, median = +0.1901, IQR = 0.5159, W = 6.00, p_raw = 0.1094, p_BH_FDR = 0.1458, p_Bonf = 0.4375

--- Post-Hoc Pairwise 

## Posterior Alpha-Beta Ratio Delta (PostClean_Delta_AB_Post)

In [5]:
analyse_metric(df_comp, "PostClean_Delta_AB_Post", EEG_CONDS, test_zero=True)

ANALYSIS: PostClean_Delta_AB_Post

--- Shapiro-Wilk Normality: PostClean_Delta_AB_Post ---
  eau       : W = 0.8167, p = 0.0431 (NON-NORMAL)
  the_vert  : W = 0.9371, p = 0.5833 (Normal)
  biscuit   : W = 0.6211, p = 0.0003 (NON-NORMAL)
  chocolat  : W = 0.9607, p = 0.8168 (Normal)
  [Non-Parametric Friedman (Non-Normal)] chi2(3) = 0.6000, p = 0.8964, Kendall's W = 0.0250

--- One-Sample Wilcoxon vs 0: PostClean_Delta_AB_Post ---
  eau       : mean = +0.1963, SEM = 0.1672, median = +0.1396, IQR = 0.2027, W = 9.00, p_raw = 0.2500, p_BH_FDR = 0.2500, p_Bonf = 1.0000
  the_vert  : mean = +0.3293, SEM = 0.2014, median = +0.2205, IQR = 0.5138, W = 7.00, p_raw = 0.1484, p_BH_FDR = 0.2500, p_Bonf = 0.5938
  biscuit   : mean = +0.6984, SEM = 0.4965, median = +0.1821, IQR = 0.4420, W = 5.00, p_raw = 0.0781, p_BH_FDR = 0.2500, p_Bonf = 0.3125
  chocolat  : mean = +0.4346, SEM = 0.2481, median = +0.2750, IQR = 0.9192, W = 8.00, p_raw = 0.1953, p_BH_FDR = 0.2500, p_Bonf = 0.7812

--- Post-Hoc Pair

## Engagement Index Delta (PostClean_Delta_EI)

In [6]:
analyse_metric(df_comp, "PostClean_Delta_EI", EEG_CONDS, test_zero=True)

ANALYSIS: PostClean_Delta_EI

--- Shapiro-Wilk Normality: PostClean_Delta_EI ---
  eau       : W = 0.9599, p = 0.8095 (Normal)
  the_vert  : W = 0.7951, p = 0.0253 (NON-NORMAL)
  biscuit   : W = 0.8644, p = 0.1328 (Normal)
  chocolat  : W = 0.6223, p = 0.0003 (NON-NORMAL)
  [Non-Parametric Friedman (Non-Normal)] chi2(3) = 0.3000, p = 0.9600, Kendall's W = 0.0125

--- One-Sample Wilcoxon vs 0: PostClean_Delta_EI ---
  eau       : mean = -0.0865, SEM = 0.0469, median = -0.1152, IQR = 0.1398, W = 6.00, p_raw = 0.1094, p_BH_FDR = 0.2188, p_Bonf = 0.4375
  the_vert  : mean = -0.0801, SEM = 0.1055, median = +0.0019, IQR = 0.2058, W = 16.00, p_raw = 0.8438, p_BH_FDR = 0.8438, p_Bonf = 1.0000
  biscuit   : mean = -0.0930, SEM = 0.0416, median = -0.0843, IQR = 0.0328, W = 6.00, p_raw = 0.1094, p_BH_FDR = 0.2188, p_Bonf = 0.4375
  chocolat  : mean = +0.0806, SEM = 0.2027, median = -0.0577, IQR = 0.0991, W = 9.00, p_raw = 0.2500, p_BH_FDR = 0.3333, p_Bonf = 1.0000

--- Post-Hoc Pairwise Wilcoxon:

## Survey Delicious Score (Survey_Delicious)

In [7]:
analyse_metric(df_comp, "Survey_Delicious", SURVEY_CONDS, test_zero=False)

ANALYSIS: Survey_Delicious

--- Shapiro-Wilk Normality: Survey_Delicious ---
  the_vert  : W = 0.8659, p = 0.1375 (Normal)
  biscuit   : W = 0.8148, p = 0.0411 (NON-NORMAL)
  chocolat  : W = 0.8832, p = 0.2020 (Normal)
  [Non-Parametric Friedman (Non-Normal)] chi2(2) = 10.5714, p = 0.0051, Kendall's W = 0.6607

--- Post-Hoc Pairwise Wilcoxon: Survey_Delicious ---
  the_vert   vs biscuit   : mean_diff = +0.0625, med_diff = +0.0000, W = 10.00, p_raw = 1.0000, p_BH_FDR = 1.0000, p_Bonf = 1.0000
  the_vert   vs chocolat  : mean_diff = -2.0000, med_diff = -2.0000, W = 0.00, p_raw = 0.0312, p_BH_FDR = 0.0469, p_Bonf = 0.0938
  biscuit    vs chocolat  : mean_diff = -2.0625, med_diff = -2.0000, W = 0.00, p_raw = 0.0078, p_BH_FDR = 0.0234, p_Bonf = 0.0234


## Survey Happy Score (Survey_Happy)

In [8]:
analyse_metric(df_comp, "Survey_Happy", SURVEY_CONDS, test_zero=False)

ANALYSIS: Survey_Happy

--- Shapiro-Wilk Normality: Survey_Happy ---
  the_vert  : W = 0.9562, p = 0.7728 (Normal)
  biscuit   : W = 0.7976, p = 0.0270 (NON-NORMAL)
  chocolat  : W = 0.9116, p = 0.3657 (Normal)
  [Non-Parametric Friedman (Non-Normal)] chi2(2) = 9.2500, p = 0.0098, Kendall's W = 0.5781

--- Post-Hoc Pairwise Wilcoxon: Survey_Happy ---
  the_vert   vs biscuit   : mean_diff = +0.1250, med_diff = +0.0000, W = 17.00, p_raw = 1.0000, p_BH_FDR = 1.0000, p_Bonf = 1.0000
  the_vert   vs chocolat  : mean_diff = -2.1250, med_diff = -2.0000, W = 4.00, p_raw = 0.0547, p_BH_FDR = 0.0820, p_Bonf = 0.1641
  biscuit    vs chocolat  : mean_diff = -2.2500, med_diff = -2.0000, W = 0.00, p_raw = 0.0078, p_BH_FDR = 0.0234, p_Bonf = 0.0234


## Survey Calm Score (Survey_Calm)

In [9]:
analyse_metric(df_comp, "Survey_Calm", SURVEY_CONDS, test_zero=False)

ANALYSIS: Survey_Calm

--- Shapiro-Wilk Normality: Survey_Calm ---
  the_vert  : W = 0.8511, p = 0.0978 (Normal)
  biscuit   : W = 0.7823, p = 0.0185 (NON-NORMAL)
  chocolat  : W = 0.8774, p = 0.1780 (Normal)
  [Non-Parametric Friedman (Non-Normal)] chi2(2) = 4.5600, p = 0.1023, Kendall's W = 0.2850

--- Post-Hoc Pairwise Wilcoxon: Survey_Calm ---
  the_vert   vs biscuit   : mean_diff = +0.3125, med_diff = +0.7500, W = 10.00, p_raw = 0.5312, p_BH_FDR = 0.5312, p_Bonf = 1.0000
  the_vert   vs chocolat  : mean_diff = +0.9375, med_diff = +1.0000, W = 5.00, p_raw = 0.1875, p_BH_FDR = 0.3750, p_Bonf = 0.5625
  biscuit    vs chocolat  : mean_diff = +0.6250, med_diff = +0.0000, W = 1.00, p_raw = 0.2500, p_BH_FDR = 0.3750, p_Bonf = 0.7500


## Pre-Touch vs Post-Swallow Frontal Alpha Surge

In [10]:
run_band_surge_test(df_stat, "Alpha_FC_uV2", "Frontal Alpha Surge (Alpha_FC_uV2)", EEG_CONDS)

PRE-TOUCH vs POST-SWALLOW: Frontal Alpha Surge (Alpha_FC_uV2)
  eau        (N=8): med_pre=6.9681, med_post=7.7577, median_change=+7.6931%, W=11.00, p_raw=0.3828, p_BH_FDR=0.3828, p_Bonf=1.0000
  the_vert   (N=8): med_pre=7.0082, med_post=9.1598, median_change=+26.7685%, W=2.00, p_raw=0.0234, p_BH_FDR=0.0469, p_Bonf=0.0938
  biscuit    (N=8): med_pre=9.4980, med_post=11.5132, median_change=+30.0446%, W=7.00, p_raw=0.1484, p_BH_FDR=0.1979, p_Bonf=0.5938
  chocolat   (N=8): med_pre=6.3766, med_post=11.1638, median_change=+30.7463%, W=0.00, p_raw=0.0078, p_BH_FDR=0.0312, p_Bonf=0.0312


## Pre-Touch vs Post-Swallow Frontal Theta Surge

In [11]:
run_band_surge_test(df_stat, "Theta_FC_uV2", "Frontal Theta Surge (Theta_FC_uV2)", EEG_CONDS)

PRE-TOUCH vs POST-SWALLOW: Frontal Theta Surge (Theta_FC_uV2)
  eau        (N=8): med_pre=6.0093, med_post=6.2575, median_change=+4.4215%, W=15.00, p_raw=0.7422, p_BH_FDR=0.7422, p_Bonf=1.0000
  the_vert   (N=8): med_pre=7.0564, med_post=11.6849, median_change=+7.2609%, W=7.00, p_raw=0.1484, p_BH_FDR=0.5938, p_Bonf=0.5938
  biscuit    (N=8): med_pre=7.9998, med_post=10.4168, median_change=+6.0379%, W=11.00, p_raw=0.3828, p_BH_FDR=0.7422, p_Bonf=1.0000
  chocolat   (N=8): med_pre=7.5243, med_post=9.0721, median_change=+2.0766%, W=14.00, p_raw=0.6406, p_BH_FDR=0.7422, p_Bonf=1.0000


## Pre-Touch vs Post-Swallow Frontal Beta Surge

In [12]:
run_band_surge_test(df_stat, "Beta_FC_uV2", "Frontal Beta Surge (Beta_FC_uV2)", EEG_CONDS)

PRE-TOUCH vs POST-SWALLOW: Frontal Beta Surge (Beta_FC_uV2)
  eau        (N=8): med_pre=9.9294, med_post=10.3459, median_change=+0.8440%, W=14.00, p_raw=0.6406, p_BH_FDR=0.8542, p_Bonf=1.0000
  the_vert   (N=8): med_pre=11.1859, med_post=12.1585, median_change=+8.2805%, W=5.00, p_raw=0.0781, p_BH_FDR=0.3125, p_Bonf=0.3125
  biscuit    (N=8): med_pre=10.7131, med_post=12.7052, median_change=-2.6349%, W=18.00, p_raw=1.0000, p_BH_FDR=1.0000, p_Bonf=1.0000
  chocolat   (N=8): med_pre=8.2308, med_post=9.3111, median_change=+1.8221%, W=11.00, p_raw=0.3828, p_BH_FDR=0.7656, p_Bonf=1.0000


## Spearman Rank Correlations Between Survey Metrics and EEG Deltas

In [13]:
run_spearman_correlations(
    df_comp,
    ["Survey_Delicious", "Survey_Happy", "Survey_Calm"],
    ["PostClean_Delta_FAA_F4_F3", "PostClean_Delta_AB_FC", "PostClean_Delta_EI"],
)

SPEARMAN RANK CORRELATIONS: SURVEY METRICS vs EEG DELTAS (Non-Water Conditions)
  Survey_Delicious  vs PostClean_Delta_FAA_F4_F3 (N=24): rho = -0.3515, p_raw = 0.0921, p_BH_FDR = 0.4146, p_Bonf = 0.8293
  Survey_Delicious  vs PostClean_Delta_AB_FC    (N=24): rho = -0.1466, p_raw = 0.4941, p_BH_FDR = 0.7677, p_Bonf = 1.0000
  Survey_Delicious  vs PostClean_Delta_EI       (N=24): rho = -0.0635, p_raw = 0.7680, p_BH_FDR = 0.8640, p_Bonf = 1.0000
  Survey_Happy      vs PostClean_Delta_FAA_F4_F3 (N=24): rho = -0.5374, p_raw = 0.0068, p_BH_FDR = 0.0609, p_Bonf = 0.0609
  Survey_Happy      vs PostClean_Delta_AB_FC    (N=24): rho = -0.1408, p_raw = 0.5118, p_BH_FDR = 0.7677, p_Bonf = 1.0000
  Survey_Happy      vs PostClean_Delta_EI       (N=24): rho = +0.1686, p_raw = 0.4308, p_BH_FDR = 0.7677, p_Bonf = 1.0000
  Survey_Calm       vs PostClean_Delta_FAA_F4_F3 (N=24): rho = -0.0878, p_raw = 0.6834, p_BH_FDR = 0.8640, p_Bonf = 1.0000
  Survey_Calm       vs PostClean_Delta_AB_FC    (N=24): rho = -

## Export All Statistical Results to TXT

In [14]:
with open(OUTPUT_TXT_PATH, "w", encoding="utf-8") as log_file, redirect_stdout(log_file):
    for metric in [
        "PostClean_Delta_FAA_F4_F3",
        "PostClean_Delta_AB_FC",
        "PostClean_Delta_AB_Post",
        "PostClean_Delta_EI",
    ]:
        analyse_metric(df_comp, metric, EEG_CONDS, test_zero=True)

    for metric in ["Survey_Delicious", "Survey_Happy", "Survey_Calm"]:
        analyse_metric(df_comp, metric, SURVEY_CONDS, test_zero=False)

    run_band_surge_test(df_stat, "Alpha_FC_uV2", "Frontal Alpha Surge (Alpha_FC_uV2)", EEG_CONDS)
    run_band_surge_test(df_stat, "Theta_FC_uV2", "Frontal Theta Surge (Theta_FC_uV2)", EEG_CONDS)
    run_band_surge_test(df_stat, "Beta_FC_uV2", "Frontal Beta Surge (Beta_FC_uV2)", EEG_CONDS)

    run_spearman_correlations(
        df_comp,
        ["Survey_Delicious", "Survey_Happy", "Survey_Calm"],
        ["PostClean_Delta_FAA_F4_F3", "PostClean_Delta_AB_FC", "PostClean_Delta_EI"],
    )

print(f"Exported statistical results to: {OUTPUT_TXT_PATH}")

Exported statistical results to: /home/ajfedora/Documents/fyp_research/process_eeg_dataset/results/statistical_analysis_results.txt
